# Baixar imagens do S3

Autor:  Viviane da Rosa Sommer

Atualizado: 19/08/2025

Objetivo: O objetivo desse código é automatizar o download de todos os arquivos de uma pasta específica em um bucket S3 da AWS para um diretório local, mantendo a estrutura de diretórios original. Ele utiliza credenciais armazenadas em um arquivo de configuração para autenticar no S3 e faz o download recursivo dos arquivos de uma pasta definida, facilitando a sincronização de dados entre o S3 e o ambiente local.

## Importações necessárias

In [ ]:
import boto3
import os
import json
import pandas as pd
from tqdm import tqdm
from typing import Optional

## Declaração de Constantes e Modelos

In [ ]:
with open("config.json", 'r') as f:
    config = json.load(f)
data_config = config[0]
s3 = boto3.client(
    's3',
    aws_access_key_id=data_config["aws_access_id"],
    aws_secret_access_key=data_config["secret_key"]
)

df = pd.read_csv('os_caminho_completo.csv')
extensoes = (".png", ".jpg", ".jpeg")

## Funções necessárias

In [ ]:
def download_s3_photos_flat(bucket_name: str, s3_folder: str, local_dir: str, s3_client, extensions: tuple) -> bool:
    """
    Downloads all photo files from an S3 bucket under a folder containing 'FOTOS' in its path, saving them flatly in a local directory.

    Args:
        bucket_name (str): Name of the S3 bucket.
        s3_folder (str): Prefix/folder in the bucket to search for photos.
        local_dir (str): Local directory to save downloaded photos.
        s3_client: Boto3 S3 client instance.
        extensions (tuple): Tuple of allowed file extensions (e.g., ('.jpg', '.png')).

    Returns:
        bool: True if at least one image was downloaded, otherwise False.
    """
    paginator = s3_client.get_paginator('list_objects_v2')
    photos_prefix: Optional[str] = None
    for page in paginator.paginate(Bucket=bucket_name, Prefix=s3_folder):
        if 'Contents' not in page:
            continue
        for obj in page['Contents']:
            key = obj['Key']
            parts = key[len(s3_folder):].split('/')
            for i in range(len(parts)):
                folder_candidate = '/'.join(parts[:i+1])
                if 'FOTOS' in folder_candidate.upper():
                    photos_prefix = f"{s3_folder}{folder_candidate}".replace('\\', '/')
                    break
            if photos_prefix:
                break
        if photos_prefix:
            break
    if not photos_prefix:
        return False
    images_downloaded = 0
    for page in paginator.paginate(Bucket=bucket_name, Prefix=photos_prefix):
        if 'Contents' not in page:
            continue
        for obj in page['Contents']:
            s3_file_path = obj['Key']
            if not s3_file_path.lower().endswith(extensions):
                continue
            local_file_path = os.path.join(local_dir, os.path.basename(s3_file_path))
            if images_downloaded == 0 and not os.path.exists(local_dir):
                os.makedirs(local_dir, exist_ok=True)
            s3_client.download_file(bucket_name, s3_file_path, local_file_path)
            images_downloaded += 1
    return images_downloaded > 0

## Baixar vídeos

In [ ]:
for _, row in tqdm(df.iterrows(), total=len(df)):
    bucket_name = row['Caminho_Completo'].split("/")[0]
    s3_folder = row['Caminho_Completo'].replace(row['Caminho_Completo'].split("/")[0] + "/", "")
    local_dir = 'originais/' + str(row['ID']) + "_" + str(row['OS'])
    baixou = download_s3_photos_flat(bucket_name, s3_folder, local_dir)
    if not baixou:
        print(local_dir)